<a href="https://colab.research.google.com/github/Anhelina4/ab-test-significance/blob/main/A_B_Testing_Statistical_Significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Significance calculation**

Metrics and Segments are defined in a config dict.

1.  **Overall, per test.** Each metric is compared between test groups with a two-proportion z-test (α = 0.05, two-sided). No correction is applied at this level. The four metrics are funnel stages of the same test

2. **Segments.** The same calculation per test, broken down by segment. A segment is tested only if each group has at least 10 conversions and 10 non-conversions – the success-failure condition for the z-test. Below that, p-values are unreliable, so the segment is marked `is_testable = False` and gets no verdict.

3. **BH correction.** For multiple tests comparison to limit false positives, segment p-values are corrected with Benjamini-Hochberg (FDR). A family is one test × one breakdown dimension (segment). Each family is corrected separately; `n_family` records how many comparisons are in the family. Untestable segments are excluded before correction.

4. **Confidence intervals.** 95% confidence intervals on the absolute difference in conversion rates, uncorrected. The interval shows how precisely the conversion difference was measured in that segment.

**Output.** `verdict` is the decision field: significant positive, significant negative, not significant, insufficient data. `p_value`, `p_adj` and `n_family` are included for transparency.


In [ ]:
from google.colab import userdata
from google.colab import drive
from google.colab import userdata
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from  scipy import stats
from typing import Any

drive.mount('/content/drive') # add your Google Drive path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive_path = userdata.get('drive_export_path')

df = pd.read_csv(f"{drive_path}/data_source.csv") # add your file path
df.head()

,date,test,test_group,device,continent,country,channel,event_name,value
0,2020-11-01,1,2,mobile,Europe,Lithuania,Paid Search,scroll,2
1,2020-11-01,1,1,mobile,Europe,Slovenia,Direct,user_engagement,30
2,2020-11-02,1,2,desktop,Africa,Tunisia,Paid Search,session_start,1
3,2020-11-02,2,1,desktop,Americas,Guatemala,Paid Search,first_visit,1
4,2020-11-02,2,1,mobile,Asia,Iraq,Paid Search,session_start,1


In [ ]:
METRICS = {
    'add_payment_info': {
        'metric': 'add_payment_info',
        "numerator": "add_payment_info",
        "denominator": "session"
    },
    'add_shipping_info': {
        'metric': 'add_shipping_info',
        "numerator": "add_shipping_info",
        "denominator": "session"
    },
    'begin_checkout': {
        'metric': 'begin_checkout',
        "numerator": "begin_checkout",
        "denominator": "session"
    },
    'new_accounts': {
        'metric': 'new_accounts',
        "numerator": "new_accounts",
        "denominator": "session"
    },
}
ALPHA = 0.05
Z_CRITICAL = stats.norm.ppf(1 - ALPHA / 2)
SEGMENTS = ['device', 'channel', 'country']
MIN_EVENTS = 10

In [ ]:
def calc_confidence_interval(num_1: float, num_2: float, den_1: int, den_2: int):
    conv_1 = num_1 / den_1
    conv_2 = num_2 / den_2
    conversion_diff = conv_2 - conv_1

    se_diff = np.sqrt(
        conv_1 * (1 - conv_1) / den_1
      + conv_2 * (1 - conv_2) / den_2
    )

    ci_lower =conversion_diff -  Z_CRITICAL * se_diff
    ci_upper =conversion_diff +  Z_CRITICAL * se_diff

    return conversion_diff, ci_lower,ci_upper

def calc_conversion_significance(num_1: float, num_2: float, den_1:float, den_2:float):
    count = np.array([num_2, num_1])
    nobs = np.array([den_2, den_1])
    z_stat, p_value = proportions_ztest(count, nobs)

    return z_stat, p_value



def calc_significance(df: pd.DataFrame, metrics: dict, test: int = 1, segment_type: Any ="total", segment_value: Any ="all" ):
    result = []

    for metric, cfg in metrics.items():
        numerator = cfg['numerator']
        denominator = cfg['denominator']

        num_1_sum = df[(df['test'] == test) & (df['test_group'] == 1) & (df['event_name'] == numerator)]['value'].sum()
        den_1_sum = df[(df['test'] == test) & (df['test_group'] == 1) & (df['event_name'] == denominator)]['value'].sum()

        num_2_sum = df[(df['test'] == test) & (df['test_group'] == 2) & (df['event_name'] == numerator)]['value'].sum()
        den_2_sum = df[(df['test'] == test) & (df['test_group'] == 2) & (df['event_name'] == denominator)]['value'].sum()


        non_conversions_1 = den_1_sum - num_1_sum
        non_conversions_2 = den_2_sum - num_2_sum

        is_testable = num_1_sum >= MIN_EVENTS and num_2_sum >= MIN_EVENTS and non_conversions_1 >= MIN_EVENTS and non_conversions_2 >= MIN_EVENTS

        row = {
            'metric': metric,
            'numerator': numerator,
            'denominator': denominator,
            'segment_type': segment_type,
            'segment_value': segment_value,
            'test': test,
            'tg_1_numerator': num_1_sum,
            'tg_1_denominator': den_1_sum,
            'tg_2_numerator': num_2_sum,
            'tg_2_denominator': den_2_sum,
        }
        if is_testable:
            conv_1 = num_1_sum / den_1_sum
            conv_2 = num_2_sum / den_2_sum

            relative_uplift = (conv_2 - conv_1) / conv_1
            z_stat, p_value = calc_conversion_significance(num_1_sum, num_2_sum, den_1_sum, den_2_sum)

            conversion_diff, ci_lower, ci_upper = calc_confidence_interval(num_1_sum, num_2_sum, den_1_sum, den_2_sum)


            row.update({
                'is_testable': True,
                'conversion_1': conv_1,
                'conversion_2': conv_2,
                'relative_uplift': relative_uplift,
                'conversion_diff': conversion_diff,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'z_stat': z_stat,
                'p_value': p_value,
                'is_significant': True if p_value <= ALPHA else False
            })
        else:
            row.update({
                'is_testable': False,
                'conversion_1': np.nan,
                'conversion_2': np.nan,
                'relative_uplift': np.nan,
                'conversion_diff': np.nan,
                'ci_lower': np.nan,
                'ci_upper': np.nan,
                'z_stat': np.nan,
                'p_value': np.nan,
                'is_significant': np.nan
            })
        result.append(row)

    return result



def generate_metrics_significance_df(df: pd.DataFrame, metrics:dict):
    result = []

    for test in sorted(df['test'].unique()):
        test_df = df[df['test'] == test]
        test_metrics = calc_significance(df=test_df, metrics=METRICS, test=test)

        result.extend(test_metrics)

        for segment_type in SEGMENTS:
            for segment_value in test_df[segment_type].unique():

                seg_df = test_df[test_df[segment_type] == segment_value]
                seg_metrics = calc_significance(df=seg_df, metrics=METRICS, test=test, segment_type=segment_type, segment_value=segment_value)

                result.extend(seg_metrics)

    return pd.DataFrame(result)

result_df = generate_metrics_significance_df(df=df, metrics=METRICS)

display(result_df)

,metric,numerator,denominator,segment_type,segment_value,test,tg_1_numerator,tg_1_denominator,tg_2_numerator,tg_2_denominator,is_testable,conversion_1,conversion_2,relative_uplift,conversion_diff,ci_lower,ci_upper,z_stat,p_value,is_significant
0,add_payment_info,add_payment_info,session,total,all,1,1988,45362,2229,45193,True,0.043825,0.049322,0.125420,0.005497,0.002752,0.008241,3.924884,0.000087,True
1,add_shipping_info,add_shipping_info,session,total,all,1,3034,45362,3221,45193,True,0.066884,0.071272,0.065605,0.004388,0.001085,0.007691,2.603571,0.009226,True
2,begin_checkout,begin_checkout,session,total,all,1,3784,45362,4021,45193,True,0.083418,0.088974,0.066606,0.005556,0.001900,0.009212,2.978783,0.002894,True
3,new_accounts,new_accounts,session,total,all,1,3823,45362,3681,45193,True,0.084278,0.081451,-0.033543,-0.002827,-0.006418,0.000764,-1.542883,0.122859,False
4,add_payment_info,add_payment_info,session,device,mobile,1,810,17896,942,17767,True,0.045262,0.053020,0.171407,0.007758,0.003271,0.012245,3.389330,0.000701,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,new_accounts,new_accounts,session,country,United States,4,3983,46178,3845,46030,True,0.086253,0.083532,-0.031543,-0.002721,-0.006319,0.000877,-1.482041,0.138329,False
1868,add_payment_info,add_payment_info,session,country,Vietnam,4,13,428,9,438,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1869,add_shipping_info,add_shipping_info,session,country,Vietnam,4,19,428,14,438,True,0.044393,0.031963,-0.279981,-0.012429,-0.037966,0.013108,-0.955161,0.339496,False
1870,begin_checkout,begin_checkout,session,country,Vietnam,4,49,428,34,438,True,0.114486,0.077626,-0.321964,-0.036860,-0.076076,0.002355,-1.842291,0.065433,False


In [ ]:
result_df['family_id'] = result_df['test'].astype(str) + "_" + result_df['segment_type']

bh_mask = (result_df['segment_type'] != 'total') & (result_df['is_testable'] == True)

result_df.loc[bh_mask, 'p_adj'] = (
    result_df[bh_mask]
    .groupby('family_id')['p_value']
    .transform(lambda family_p_values: multipletests(family_p_values, method='fdr_bh')[1])
)
result_df['is_BH_significant'] = (
    (result_df.loc[bh_mask, 'p_adj'] <= ALPHA)
    .where(result_df['is_testable'])
)

result_df.loc[bh_mask, 'n_family'] = (
    result_df[bh_mask]
    .groupby('family_id')['family_id']
    .transform(lambda family: family.size)
)

result_df['verdict_p_value'] = result_df['p_adj'].fillna(result_df['p_value'])

result_df['verdict'] = np.where(
    ~result_df['is_testable'],
    'insufficient data',
    np.where(
        result_df['verdict_p_value'] > ALPHA,
        'not significant',
        np.where(
            result_df['relative_uplift'] > 0,
            'significant positive',
            'significant negative'
        )
    )
)

display(result_df)

,metric,numerator,denominator,segment_type,segment_value,test,tg_1_numerator,tg_1_denominator,tg_2_numerator,tg_2_denominator,...,ci_upper,z_stat,p_value,is_significant,family_id,p_adj,is_BH_significant,n_family,verdict_p_value,verdict
0,add_payment_info,add_payment_info,session,total,all,1,1988,45362,2229,45193,...,0.008241,3.924884,0.000087,True,1_total,NaN,NaN,NaN,0.000087,significant positive
1,add_shipping_info,add_shipping_info,session,total,all,1,3034,45362,3221,45193,...,0.007691,2.603571,0.009226,True,1_total,NaN,NaN,NaN,0.009226,significant positive
2,begin_checkout,begin_checkout,session,total,all,1,3784,45362,4021,45193,...,0.009212,2.978783,0.002894,True,1_total,NaN,NaN,NaN,0.002894,significant positive
3,new_accounts,new_accounts,session,total,all,1,3823,45362,3681,45193,...,0.000764,-1.542883,0.122859,False,1_total,NaN,NaN,NaN,0.122859,not significant
4,add_payment_info,add_payment_info,session,device,mobile,1,810,17896,942,17767,...,0.012245,3.389330,0.000701,True,1_device,0.002803,True,12.0,0.002803,significant positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,new_accounts,new_accounts,session,country,United States,4,3983,46178,3845,46030,...,0.000877,-1.482041,0.138329,False,4_country,0.417219,False,187.0,0.417219,not significant
1868,add_payment_info,add_payment_info,session,country,Vietnam,4,13,428,9,438,...,NaN,NaN,NaN,NaN,4_country,NaN,NaN,NaN,NaN,insufficient data
1869,add_shipping_info,add_shipping_info,session,country,Vietnam,4,19,428,14,438,...,0.013108,-0.955161,0.339496,False,4_country,0.668916,False,187.0,0.668916,not significant
1870,begin_checkout,begin_checkout,session,country,Vietnam,4,49,428,34,438,...,0.002355,-1.842291,0.065433,False,4_country,0.235306,False,187.0,0.235306,not significant


In [ ]:
result_df.to_csv(f'{drive_path}/ab_test_results.csv')